# 🧠 AI Code Review System

Multi-agent pipeline: **Claude** (generate/fix/optimize) → **GPT** (validate/review)

**Flow:** `Code Understanding → Technical Review → Quality Review → Optimize → Chat Refine`

In [1]:
from dotenv import load_dotenv
from agents import Runner, Agent, trace
from openai import OpenAI
import re
import os
import json

load_dotenv(override=True)


True

## 🔧 Helper Utilities

In [2]:
def clean_json_output(text: str) -> str:
    """Strip markdown code fences from LLM JSON responses."""
    text = text.strip()
    if text.startswith("```"):
        lines = text.split("\n")
        # drop first line (```json or ```) and last line (```)
        text = "\n".join(lines[1:-1]).strip()
    return text


def extract_json(text: str) -> str:
    """Extract first JSON object from text, even if surrounded by prose."""
    text = clean_json_output(text)
    match = re.search(r"\{.*\}", text, re.DOTALL)
    return match.group(0) if match else text


def safe_json_parse(text: str) -> dict | None:
    """Parse JSON safely — tries clean → extract → fail gracefully."""
    for fn in (clean_json_output, extract_json):
        try:
            return json.loads(fn(text))
        except json.JSONDecodeError:
            pass
    print("❌ JSON parse failed. Raw output:\n", text)
    return None


## 🟡 Agent 1 — Code Understanding (Claude)

Identifies language, summarises the problem, extracts complexity.

In [3]:
UNDERSTANDER_SYSTEM = """
You are an experienced developer specialized in understanding code written by others.

Your job is to analyze code and return structured JSON with:

- programming_language_used  (Python, Java, C++, Go, Kotlin, Javascript, etc.)
- problem_summary
- approach
- key_constructs
- complexity
    - time
    - space
- confidence  (0 to 1)

Rules:
- Be concise
- Do NOT explain anything outside JSON
- Always return valid JSON

-- IMPORTANT --
Response format:
{
    "programming_language_used": "...",
    "problem_summary": "...",
    "approach": "...",
    "key_constructs": ["..."],
    "complexity": {
        "time": "...",
        "space": "..."
    },
    "confidence": 0.0
}
"""


def run_code_understander(code: str) -> dict:
    client = OpenAI(
        api_key=os.environ.get("ANTHROPIC_API_KEY"),
        base_url="https://api.anthropic.com/v1/",
    )
    user_message = f"Analyze the following code:\n\nCode:\n{code}"

    response = client.chat.completions.create(
        model="claude-sonnet-4-6",
        messages=[
            {"role": "system", "content": UNDERSTANDER_SYSTEM},
            {"role": "user",   "content": user_message},
        ],
    )

    return safe_json_parse(response.choices[0].message.content)


## 🔵 Agent 2 — Technical Review + Fix (Claude)

Detects bugs, edge cases, complexity issues, and produces corrected code.

In [ ]:
CORRECTOR_SYSTEM = """
You are a senior software engineer and expert code reviewer.

Your job is NOT to understand the code from scratch.
You will be provided with:
  1. The original code
  2. A structured analysis from a previous agent

Use this to perform a deep technical review and improve the code.

---

Your tasks:
  1. Validate correctness
  2. Identify logical bugs
  3. Detect missing edge case handling
  4. Analyze time and space complexity
  5. Suggest optimizations
  6. Recommend better tools / libraries (unbiased)
  7. Provide corrected code (minimal fix — keep structure similar)

---

Return ONLY valid JSON. Do NOT wrap in ```json or ```.

FORMAT:
{
  "correctness": "Correct | Partially Correct | Incorrect",
  "bugs": ["..."],
  "edge_cases": ["..."],
  "complexity": { "time": "...", "space": "..." },
  "optimizations": ["..."],
  "improved_approach": "...",
  "tools_recommendation": [
    { "current": "...", "suggested": "...", "reason": "..." }
  ],
  "corrected_code": "...",
  "confidence": 0.0
}

STRICT RULES:
- Be critical and precise (like a real interviewer)
- corrected_code must be valid and runnable
- If no tool improvement needed, return []
- confidence must be 0-1
"""


def run_codeCorrector(code: str, understanding: dict) -> dict:
    client = OpenAI(
        api_key=os.environ.get("ANTHROPIC_API_KEY"),
        base_url="https://api.anthropic.com/v1/",
    )
    user_message = (
        f"Code understanding from previous agent:\n{understanding}\n\n"
        f"Code:\n{code}"
    )

    response = client.chat.completions.create(
        model="claude-opus-4-6",
        messages=[
            {"role": "system", "content": CORRECTOR_SYSTEM},
            {"role": "user",   "content": user_message},
        ],
    )

    return safe_json_parse(response.choices[0].message.content)


## 🟢 Agent 3 — Code Quality & Overall Review (GPT)

Evaluates readability, maintainability, best practices, and production readiness.

In [ ]:
QUALITY_SYSTEM = """
You are a senior software engineer performing a professional code review.

Your job is to evaluate code quality, maintainability, and production readiness.

You will be provided with:
  1. Original code
  2. Code understanding analysis
  3. Technical review from another agent

---

Your tasks:
  1. Evaluate readability and clarity
  2. Identify code quality issues
  3. Assess maintainability and scalability
  4. Check adherence to best practices
  5. Suggest improvements for production-level code

---

Return ONLY valid JSON. Do NOT wrap in ```json or ```.

FORMAT:
{
  "readability_score": 0,
  "code_quality_issues": ["..."],
  "maintainability_issues": ["..."],
  "best_practice_violations": ["..."],
  "strengths": ["..."],
  "improvement_suggestions": ["..."],
  "production_readiness": {
    "status": "Low | Medium | High",
    "issues": ["..."]
  },
  "final_summary": "...",
  "confidence": 0.0
}

STRICT RULES:
- readability_score: 0-10
- Be critical and realistic
- final_summary: 2-3 lines
- confidence: 0-1
"""

# GPT-based quality agent (OpenAI Agents SDK)
code_quality_agent = Agent(
    name="Code Quality Reviewer",
    instructions=QUALITY_SYSTEM,
    model="gpt-5-mini",
)


async def run_code_quality_agent(code: str, understanding: dict, technical_review: dict) -> dict | None:
    input_text = (
        f"Code:\n{code}\n\n"
        f"Understanding:\n{understanding}\n\n"
        f"Technical Review:\n{technical_review}"
    )

    with trace("CodeQualityAgent"):
        result = await Runner.run(code_quality_agent, input=input_text)

    # FIX: use safe_json_parse so markdown-fenced GPT output is handled correctly
    return safe_json_parse(result.final_output)


## 🔘 Agent 4 — GPT Validator (used inside feedback loops)

Checks whether optimized / chat-refined code is correct and runnable.

In [ ]:
VALIDATOR_SYSTEM = """
You are a strict code validator.

You will receive code and must check:
  - Correctness and logical validity
  - Whether requested improvements were applied
  - Whether the code is runnable

Return ONLY valid JSON. Do NOT wrap in ```json or ```.

FORMAT:
{
  "valid": true,
  "issues": ["..."],
  "feedback": "..."
}
"""

# FIX: gpt_reviewer_agent was missing entirely — caused NameError in both
#      optimize_with_validation and chat_with_validation
gpt_reviewer_agent = Agent(
    name="GPT Code Validator",
    instructions=VALIDATOR_SYSTEM,
    model="gpt-5-mini",
)


## 🔁 Full Initial Pipeline

`run_full_code_review` runs all three agents in sequence and returns a combined result dict.

In [7]:
async def run_full_code_review(code: str) -> dict:
    print("🟡 Running Code Understanding Agent...")
    understanding = run_code_understander(code)
    print("✅ Understanding Done\n")

    print("🔵 Running Technical Review Agent...")
    technical_review = run_codeCorrector(code, understanding)
    print("✅ Technical Review Done\n")

    print("🟢 Running Code Quality Agent...")
    quality_review = await run_code_quality_agent(code, understanding, technical_review)
    print("✅ Quality Review Done\n")

    return {
        "understanding":    understanding,
        "technical_review": technical_review,
        "quality_review":   quality_review,
    }


### 🧪 Demo — run the full pipeline on an LRU Cache implementation

In [8]:
demo_code = """
class LRUCache:
    def __init__(self, capacity):
        self.capacity = capacity
        self.cache = {}
        self.order = []

    def get(self, key):
        if key in self.cache:
            self.order.remove(key)
            self.order.append(key)
            return self.cache[key]
        return -1

    def put(self, key, value):
        if key in self.cache:
            self.cache[key] = value
            self.order.remove(key)
            self.order.append(key)
        else:
            if len(self.cache) >= self.capacity:
                lru = self.order[0]
                del self.cache[lru]
                self.order.pop(0)
            self.cache[key] = value
            self.order.append(key)
"""

initial_result = await run_full_code_review(demo_code)
initial_result


🟡 Running Code Understanding Agent...
✅ Understanding Done

🔵 Running Technical Review Agent...
✅ Technical Review Done

🟢 Running Code Quality Agent...
✅ Quality Review Done



{'understanding': {'programming_language_used': 'Python',
  'problem_summary': 'Implementation of a Least Recently Used (LRU) Cache with a fixed capacity that supports get and put operations.',
  'approach': 'Uses a dictionary for O(1) key-value storage and a list to track the access order of keys. The front of the list represents the least recently used item, and the back represents the most recently used. On access or insertion, the key is moved to the end of the list. When capacity is exceeded, the first element (LRU) is evicted.',
  'key_constructs': ['Dictionary (self.cache) for fast key-value lookups',
   'List (self.order) for maintaining access order',
   'list.remove() to reposition accessed keys',
   'list.pop(0) to evict the least recently used key',
   'Capacity check before insertion to trigger eviction'],
  'complexity': {'time': 'O(n) for both get and put due to list.remove() and list.pop(0) being O(n) operations',
   'space': "O(capacity) for storing up to 'capacity' ke

## 🔘 Fully Optimized Code — Claude → GPT Validation Loop

Claude optimizes; GPT validates. Repeats up to `max_iters` times until valid.

In [9]:
OPTIMIZER_SYSTEM = """
You are an expert software engineer specializing in code optimization.

Your role is to produce a fully optimized, production-ready version of the code
using the feedback and reviews provided.

Responsibilities:
  - Apply all feedback and review suggestions first
  - Improve time and space complexity where possible
  - Use optimal data structures and algorithms
  - Ensure code is clean, readable, and maintainable
  - Apply best practices
  - Ensure correctness

Output Requirements:
  - Return ONLY valid JSON
  - Do NOT wrap in ``` or ```json

FORMAT:
{
  "optimized_code": "...",
  "changes_made": ["..."],
  "optimization_summary": "..."
}
"""


def call_claude_optimizer(user_message: str) -> str:
    """Call Claude with the optimizer system prompt and return raw text."""
    client = OpenAI(
        api_key=os.environ.get("ANTHROPIC_API_KEY"),
        base_url="https://api.anthropic.com/v1/",
    )
    response = client.chat.completions.create(
        model="claude-opus-4-6",
        messages=[
            {"role": "system", "content": OPTIMIZER_SYSTEM},  # FIX: was using wrong global 'system_message'
            {"role": "user",   "content": user_message},
        ],
    )
    return response.choices[0].message.content


async def optimize_with_validation(code: str, initial_result: dict, max_iters: int = 3) -> str:
    """Iteratively optimize code with Claude and validate with GPT."""
    feedback = None

    for i in range(max_iters):
        print(f"\n🔵 Claude Optimization Iteration {i + 1}")

        user_prompt = (
            f"Optimize this code to production level based on the following reviews.\n\n"
            f"Understanding:\n{initial_result.get('understanding')}\n\n"
            f"Technical Review:\n{initial_result.get('technical_review')}\n\n"
            f"Quality Review:\n{initial_result.get('quality_review')}\n\n"
            f"Previous Feedback (if any):\n{feedback}\n\n"
            f"Code:\n{code}"
        )

        claude_output = call_claude_optimizer(user_prompt)
        claude_json   = safe_json_parse(claude_output)
        if not claude_json:
            print("❌ Claude returned unparseable output, stopping.")
            break

        optimized_code = claude_json["optimized_code"]
        print("Changes:", claude_json.get("changes_made"))

        print("🟢 GPT Validation...")
        gpt_prompt = (
            f"Check if the optimized code satisfies:\n"
            f"- correctness\n- improvements suggested earlier\n- is runnable\n\n"
            f"Return JSON: {{\"valid\": true/false, \"issues\": [...], \"feedback\": \"...\"}}\n\n"
            f"Code:\n{optimized_code}"
        )

        # FIX: Runner.run() does NOT accept a 'model' kwarg — model lives on the Agent object
        gpt_result = await Runner.run(gpt_reviewer_agent, input=gpt_prompt)
        gpt_json   = safe_json_parse(gpt_result.final_output)

        if gpt_json and gpt_json.get("valid"):
            print("✅ Optimization Successful")
            return optimized_code

        issues = gpt_json.get("issues") if gpt_json else "parse error"
        print("❌ Needs Improvement:", issues)
        feedback = gpt_json.get("feedback") if gpt_json else ""
        code = optimized_code  # iterative refinement

    print("⚠️ Max iterations reached — returning last version")
    return optimized_code


### 🧪 Demo — Optimize the LRU Cache

In [10]:
# FIX: was called twice (Cells 39 + 40 both called optimize_with_validation)
optimized_code = await optimize_with_validation(demo_code, initial_result)
print("\n--- Optimized Code ---")
print(optimized_code)



🔵 Claude Optimization Iteration 1
Changes: ['Replaced dict + list with collections.OrderedDict for O(1) get and put operations instead of O(n)', 'Used OrderedDict.move_to_end() for O(1) access order updates, replacing O(n) list.remove() + list.append()', 'Used OrderedDict.popitem(last=False) for O(1) LRU eviction, replacing O(n) list.pop(0) + dict delete', 'Added capacity validation in __init__ to reject negative or non-integer values', 'Added early return in put() when capacity <= 0 to prevent unbounded cache growth', 'Added __slots__ to reduce per-instance memory overhead', 'Added type hints for all method signatures', 'Added comprehensive docstrings with usage example', 'Added __len__, __contains__, and __repr__ dunder methods for better usability and debugging', 'Consolidated the update path in put() to avoid code duplication']
🟢 GPT Validation...
✅ Optimization Successful

--- Optimized Code ---
from collections import OrderedDict


class LRUCache:
    """Least Recently Used (LRU

## 💬 Chat Refinement — Claude → GPT Validation Loop

User provides a custom instruction (e.g. *'Convert to Java'*, *'Make it O(n)'*). Claude applies it; GPT validates.

In [11]:
CHAT_SYSTEM = """
You are an expert software engineer helping refine and modify code based on user requests.

Responsibilities:
  - Apply the requested changes to the code
  - Keep context from previous agent reviews
  - Maintain correctness and readability
  - Ensure the code remains functional

Output Requirements:
  - Return ONLY valid JSON
  - Do NOT wrap in ``` or ```json

FORMAT:
{
  "updated_code": "...",
  "changes_made": ["..."],
  "explanation": "..."
}
"""


def call_claude_chat(user_message: str) -> str:
    """Call Claude with the chat-refinement system prompt and return raw text."""
    client = OpenAI(
        api_key=os.environ.get("ANTHROPIC_API_KEY"),
        base_url="https://api.anthropic.com/v1/",
    )
    response = client.chat.completions.create(
        model="claude-opus-4-6",
        messages=[
            {"role": "system", "content": CHAT_SYSTEM},
            {"role": "user",   "content": user_message},
        ],
    )
    return response.choices[0].message.content


async def chat_with_validation(code: str, initial_result: dict, user_query: str, max_iters: int = 3) -> str:
    """Apply user-requested changes with Claude, validate with GPT."""
    feedback = None

    for i in range(max_iters):
        print(f"\n🔵 Claude Chat Iteration {i + 1}")

        user_prompt = (
            f"Modify the code based on the user request.\n\n"
            f"User Request:\n{user_query}\n\n"
            f"Understanding:\n{initial_result.get('understanding')}\n\n"
            f"Technical Review:\n{initial_result.get('technical_review')}\n\n"
            f"Previous Feedback (if any):\n{feedback}\n\n"
            f"Code:\n{code}"
        )

        claude_output = call_claude_chat(user_prompt)
        claude_json   = safe_json_parse(claude_output)
        if not claude_json:
            print("❌ Claude returned unparseable output, stopping.")
            break

        updated_code = claude_json["updated_code"]
        print("Changes:", claude_json.get("changes_made"))

        print("🟢 GPT Validation...")
        gpt_prompt = (
            f"Check:\n- Is this code correct?\n- Is it runnable?\n"
            f"- Does it satisfy the user request: '{user_query}'?\n\n"
            f"Return JSON: {{\"valid\": true/false, \"issues\": [...], \"feedback\": \"...\"}}\n\n"
            f"Code:\n{updated_code}"
        )

        gpt_result = await Runner.run(gpt_reviewer_agent, input=gpt_prompt)
        gpt_json   = safe_json_parse(gpt_result.final_output)

        if gpt_json and gpt_json.get("valid"):
            print("✅ Chat Update Successful")
            return updated_code

        issues = gpt_json.get("issues") if gpt_json else "parse error"
        print("❌ Needs Fix:", issues)
        feedback = gpt_json.get("feedback") if gpt_json else ""
        code = updated_code

    print("⚠️ Max iterations reached — returning last version")
    return updated_code


### 🧪 Demo — Chat Refinement

In [12]:
# Example: ask Claude to rewrite the LRU Cache using OrderedDict (O(1) ops)
user_request = "Rewrite using collections.OrderedDict to achieve O(1) get and put operations"

refined_code = await chat_with_validation(demo_code, initial_result, user_request)
print("\n--- Refined Code ---")
print(refined_code)



🔵 Claude Chat Iteration 1
Changes: ['Replaced `dict` + `list` with `collections.OrderedDict` for O(1) order-aware operations.', 'Replaced `list.remove(key)` and `list.append(key)` with `OrderedDict.move_to_end(key)` for O(1) reordering on access.', 'Replaced `list.pop(0)` and `del self.cache[lru]` with `OrderedDict.popitem(last=False)` for O(1) LRU eviction.', 'Removed the separate `self.order` list since `OrderedDict` inherently maintains insertion/access order.', 'Added a guard clause `if self.capacity <= 0: return` in `put` to handle edge case where capacity is zero or negative.']
🟢 GPT Validation...
✅ Chat Update Successful

--- Refined Code ---
from collections import OrderedDict


class LRUCache:
    def __init__(self, capacity):
        self.capacity = capacity
        self.cache = OrderedDict()

    def get(self, key):
        if key in self.cache:
            self.cache.move_to_end(key)
            return self.cache[key]
        return -1

    def put(self, key, value):
     